In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
class OCRDataframe:
    def __init__(self, df): self.df = df
print('pandas:', pd.__version__, ' polars:', pl.__version__)

# Harness-only conversion helpers: the mined fixture is stored in target-side
# form, while the oracle must receive the semantically equivalent pandas form.
def _to_pandas_fixture(value):
    if isinstance(value, pl.DataFrame):
        return value.to_pandas()
    if isinstance(value, pl.Series):
        return value.to_pandas()
    if isinstance(value, list):
        return [_to_pandas_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_pandas_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_pandas_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_pandas_fixture(item) for key, item in vars(value).items()
        })
    return value

def _to_polars_fixture(value):
    if isinstance(value, pd.DataFrame):
        return pl.from_pandas(value)
    if isinstance(value, pd.Series):
        return pl.from_pandas(value)
    if isinstance(value, list):
        return [_to_polars_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_polars_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_polars_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_polars_fixture(item) for key, item in vars(value).items()
        })
    return value


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- data_assign_bbox ---
FIX_DATA_ASSIGN_BBOX_BBOX = (10, 10, 100, 50)

# --- data_filter_page ---
FIX_DATA_FILTER_PAGE_PAGE_NUMBER = 1

# --- data_groupby_agg_text ---
FIX_DATA_GROUPBY_AGG_TEXT_DF_WORDS_CONTAINED = pl.DataFrame({"parent": [0,0,1], "value": ["hello", "world", "again"], "x1": [1,2,3], "x2": [5,6,7], "y1": [1,1,3], "y2": [2,2,4]})

# --- data_merge_cross ---
FIX_DATA_MERGE_CROSS_MIN_CONFIDENCE = 0.5
FIX_DATA_MERGE_CROSS_PAGE_NUMBER = 1
FIX_DATA_MERGE_CROSS_TABLE = pl.DataFrame({"class": [1,2,3], "col": [1,2,3], "confidence": [1,2,3], "ocrx_word": [1,2,3]})
FIX_DATA_MERGE_CROSS_CELL = SimpleNamespace(x1=10,y1=10,x2=100,y2=50)
FIX_DATA_MERGE_CROSS_ID_COL = 0
FIX_DATA_MERGE_CROSS_ID_ROW = 0

self = SimpleNamespace(df=pd.DataFrame({"class": ["ocrx_word"], "x1": [0], "y1": [0], "x2": [10], "y2": [10], "value": ["hello"], "confidence": [95], "page_nb": [0], "page": [1]}))
print("✅ Fixtures loaded")
OCRDataframe = SimpleNamespace  # mock for testing


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_data_assign_bbox(bbox, df_words=None):
    if df_words is None:
        df_words = pd.DataFrame({"class":["ocrx_word"],"x1":[0],"y1":[0],"x2":[10],"y2":[10],"value":["hi"],"confidence":[90],"page_nb":[0]})
    df_words = df_words.assign(**{"x1_bbox": bbox[0],
                                  "y1_bbox": bbox[1],
                                  "x2_bbox": bbox[2],
                                  "y2_bbox": bbox[3]})
    df_words["x_left"] = df_words[["x1", "x1_bbox"]].max(axis=1)
    df_words["y_top"] = df_words[["y1", "y1_bbox"]].max(axis=1)
    df_words["x_right"] = df_words[["x2", "x2_bbox"]].min(axis=1)
    df_words["y_bottom"] = df_words[["y2", "y2_bbox"]].min(axis=1)
    return df_words

def before_data_filter_page(page_number):
    return OCRDataframe(df=self.df[self.df["page"] == page_number])
    return None

def before_data_groupby_agg_text(df_words_contained):
    df_text_parent = (df_words_contained.groupby('parent')
                      .agg(x1=("x1", np.min),
                           x2=("x2", np.max),
                           y1=("y1", np.min),
                           y2=("y2", np.max),
                           value=("value", lambda x: ' '.join(x)))
                      .sort_values(by=["y1", "x1"]))
    return df_text_parent["value"].astype(str).str.cat(sep="\n").strip() or None
    return df_text_parent

def before_data_merge_cross(min_confidence, page_number, table, cell, id_col, id_row):
    # ❌ SyntaxError: unterminated string literal (detected at line 30) (line 30)
    raise NotImplementedError("snippet has unfixable syntax")

_oracle_data_assign_bbox = before_data_assign_bbox
def before_data_assign_bbox(*args, **kwargs):
    return _oracle_data_assign_bbox(
        *[_to_pandas_fixture(value) for value in args],
        **{key: _to_pandas_fixture(value) for key, value in kwargs.items()},
    )


In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_data_assign_bbox(bbox, df_words=None):
    if df_words is None:
        df_words = pl.DataFrame({"class":["ocrx_word"],"x1":[0],"y1":[0],"x2":[10],"y2":[10],"value":["hi"],"confidence":[90],"page_nb":[0]})

    df_words = df_words.with_columns(
        [
            pl.lit(bbox[0]).alias("x1_bbox"),
            pl.lit(bbox[1]).alias("y1_bbox"),
            pl.lit(bbox[2]).alias("x2_bbox"),
            pl.lit(bbox[3]).alias("y2_bbox"),
        ]
    )
    df_words = df_words.with_columns(
        [
            pl.max_horizontal("x1", "x1_bbox").alias("x_left"),
            pl.max_horizontal("y1", "y1_bbox").alias("y_top"),
            pl.min_horizontal("x2", "x2_bbox").alias("x_right"),
            pl.min_horizontal("y2", "y2_bbox").alias("y_bottom"),
        ]
    )
    return df_words

def gen_data_filter_page(page_number):

    return OCRDataframe(df=self.df.filter(pl.col("page") == page_number))

def gen_data_groupby_agg_text(df_words_contained):

    df_text_parent = (
        df_words_contained.group_by("parent")
        .agg(
            pl.col("x1").min().alias("x1"),
            pl.col("x2").max().alias("x2"),
            pl.col("y1").min().alias("y1"),
            pl.col("y2").max().alias("y2"),
            pl.col("value").cast(pl.Utf8).implode().list.join(" ").alias("value"),
        )
        .sort(by=["y1", "x1"])
    )
    return "\n".join(df_text_parent.get_column("value").cast(pl.Utf8).to_list()).strip() or None

def gen_data_merge_cross(min_confidence, page_number, table, cell, id_col, id_row):

    df_words = self.df.filter(pl.col("class") == "ocrx_word")
    if page_number:
        df_words = df_words.filter(pl.col("page") == page_number)
    df_words = df_words.filter(pl.col("value").is_not_null() & (pl.col("confidence") >= min_confidence))

    list_cells = [
        {"row": id_row, "col": id_col, "x1_w": cell.x1, "x2_w": cell.x2, "y1_w": cell.y1, "y2_w": cell.y2}
        for id_row, row in enumerate(table.items)
        for id_col, cell in enumerate(row.items)
    ]
    df_cells = pl.DataFrame(list_cells)

    df_word_cells = df_words.join(df_cells, how="cross")

    df_word_cells = df_word_cells.with_columns(
        x_left=pl.max_horizontal("x1", "x1_w"),
        y_top=pl.max_horizontal("y1", "y1_w"),
        x_right=pl.min_horizontal("x2", "x2_w"),
        y_bottom=pl.min_horizontal("y2", "y2_w"),
    )

    df_word_cells = df_word_cells.filter(pl.col("x_right") > pl.col("x_left"))
    df_word_cells = df_word_cells.filter(pl.col("y_bottom") > pl.col("y_top"))

    df_text_parent = (
        df_word_cells
        .group_by(["row", "col", "parent"], maintain_order=True)
        .agg(value=pl.col("value").str.join(" "))
        .sort(["row", "col"])
        .group_by(["row", "col"], maintain_order=True)
        .agg(text=pl.col("value").str.join("\n").replace("", None))
    )
    return df_text_parent

def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)

# ── Test harness type adapters ─────────────────────────────────────────────
def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)


In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if hasattr(r, "df"):
        r = r.df
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: data_assign_bbox ===

# L1 smoke – generated
try:
    _r = gen_data_assign_bbox(FIX_DATA_ASSIGN_BBOX_BBOX)
    print("✅ L1 smoke gen_data_assign_bbox: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_data_assign_bbox: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_data_assign_bbox(FIX_DATA_ASSIGN_BBOX_BBOX)
    print("✅ L1 smoke before_data_assign_bbox: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_data_assign_bbox: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_data_assign_bbox(FIX_DATA_ASSIGN_BBOX_BBOX)
    _rg = gen_data_assign_bbox(FIX_DATA_ASSIGN_BBOX_BBOX)
    compare(_rb, _rg, "data_assign_bbox")
except Exception as _e:
    print(f"❌ L2 equivalence data_assign_bbox: setup error — {type(_e).__name__}: {_e}")

# L3 edge - explicit empty input exercises the non-default frame branch.
try:
    _empty = pl.DataFrame(schema={
        "class": pl.String, "x1": pl.Int64, "y1": pl.Int64,
        "x2": pl.Int64, "y2": pl.Int64, "value": pl.String,
        "confidence": pl.Int64, "page_nb": pl.Int64,
    })
    _rb = before_data_assign_bbox(FIX_DATA_ASSIGN_BBOX_BBOX, _empty)
    _rg = gen_data_assign_bbox(FIX_DATA_ASSIGN_BBOX_BBOX, _empty)
    compare(_rb, _rg, "L3 edge data_assign_bbox empty frame", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge data_assign_bbox empty frame: {type(_e).__name__}: {_e}")
